# 🧬 RontoGPT — Adaptateurs LoRA pour le cerveau **Gemma-3-1B**

**Runtime → Modifier le type d'exécution → T4 GPU**, puis *Exécuter tout* (~20-25 min).

Ce notebook est **autonome** : tout le code d'entraînement est dans les cellules
ci-dessous. Tu n'as **aucun `.py` à récupérer** du dépôt.

Depuis le 9 oct. 2026, le cerveau par défaut d'RontoGPT est
`Gemma-3-1B-it-GLM-4.7-Flash-Heretic-Uncensored-Thinking` (architecture `gemma3` :
26 couches, dim 1152, 4 têtes / 1 KV). Les 4 anciens adaptateurs Llama-3.2 sont
des deltas d'une AUTRE architecture (16 couches, dim 2048) : ils sont refusés
par la mesure de l'en-tête GGUF. Ce notebook les **ré-entraîne** pour Gemma.

| Adaptateur | Rôle |
|---|---|
| `aura-web` | journaliste factuel, répond à partir des faits |
| `aura-math` | calcul + raisonnement numérique |
| `aura-code` | expert Python commenté |
| `aura-general` | assistant clair et précis |

**Pourquoi la base `unsloth/gemma-3-1b-it` et pas le GGUF Heretic** : un dépôt
GGUF ne contient **aucun poids safetensors**, donc rien d'entraînable. Le Heretic
est un fine-tune de `google/gemma-3-1b-it` ; les deltas LoRA sont calculés sur la
**même architecture** que le cerveau GGUF, donc montables dessus. Le miroir
`unsloth` publie les mêmes poids **sans token HF ni licence à accepter**.

À la fin : 4 fichiers `.gguf` à déposer dans `adaptateurs/` sur le PC. Le montage
lit l'architecture **de l'adaptateur lui-même** — rien à configurer.


In [ ]:
# 1. Dépendances (~2 min)
# torchao : Colab embarque une version trop vieille pour peft (>0.16 requis)
# et le LoRA ne l'utilise PAS -> on le retire, peft le saute proprement.
# sentencepiece : le tokenizer Gemma en a besoin.
!pip -q uninstall -y torchao
!pip -q install peft transformers accelerate bitsandbytes gguf sentencepiece
import os
# Reduit la fragmentation du cache CUDA (recommande par PyTorch sur T4).
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
import torch
print('GPU :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'AUCUN (active le T4 !)')
print('bf16 :', torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False)


## 📚 Données

Le notebook cherche les jeux de données dans cet ordre, et **écrit toujours**
`datasets/<cat>.jsonl` (fichiers locaux > dépôt GitHub > amorce intégrée) :

1. `datasets/web.jsonl`, `math.jsonl`, `code.jsonl`, `general.jsonl` — si tu les
   déposes à la main (glisser-déposer dans Colab, ou Drive) ;
2. le dépôt GitHub (clone automatique) ;
3. un **petit jeu d'amorçage intégré** — le notebook tourne même sans rien.

Format JSONL (≥ 200 exemples recommandés pour un vrai résultat) :

```json
{"question": "...", "reponse": "..."}
{"question": "...", "reponse": "...", "contexte": "faits utiles"}      // option
{"question": "...", "reponse": "...", "pensee": "raisonnement"}        // option
```

Le champ `pensee` (option) est écrit dans le bloc de raisonnement natif
(`<think>…</think>`) : l'adaptateur apprend à penser dans la même balise que
celle que le flux d'RontoGPT montre à l'utilisateur. C'est la sortie du distillateur
de self-play (`scripts/selfplay_distille.py`).


In [ ]:
# 2. Résolution des données (le notebook tourne même sans rien uploader)
import json, subprocess
from pathlib import Path

CATEGORIES = ['web', 'math', 'code', 'general']
DEPOT = 'https://github.com/Simonc44/aura-1b.git'

AMORCE = {
    'web': [
        {'question': "Quelle est la capitale de l'Australie ?", 'reponse': 'Canberra.'},
        {'question': 'Qui a peint la Joconde ?', 'reponse': 'Léonard de Vinci.'},
        {'question': 'En quelle année est tombé le mur de Berlin ?', 'reponse': 'En 1989.'},
        {'question': 'Quelle est la planète la plus proche du Soleil ?', 'reponse': 'Mercure.'},
        {'question': "Quel est le plus long fleuve de France ?", 'reponse': 'La Loire.'},
        {'question': 'Combien de côtés a un hexagone ?', 'reponse': 'Six.'},
    ],
    'math': [
        {'question': 'Combien font 17 × 23 ?', 'reponse': '391 (17 × 23 = 391).'},
        {'question': 'Quelle est la moyenne de 4, 8 et 10 ?', 'reponse': '(4+8+10)/3 = 22/3 ≈ 7,33.'},
        {'question': 'Combien font 15 % de 240 ?', 'reponse': '36 (240 × 0,15 = 36).'},
        {'question': 'Résous : 3x + 6 = 21', 'reponse': '3x = 15, donc x = 5.'},
        {'question': "Quelle est l'aire d'un cercle de rayon 3 ?", 'reponse': 'πr² = 9π ≈ 28,27.'},
        {'question': 'Combien font 2⁸ ?', 'reponse': '256 (2⁸ = 256).'},
    ],
    'code': [
        {'question': 'Écris une fonction Python qui inverse une chaîne.',
         'reponse': 'def inverser(s):\n    return s[::-1]\n\n# inverser("aura") -> "arua"'},
        {'question': 'Écris une fonction Python qui calcule la factorielle.',
         'reponse': 'def factorielle(n):\n    r = 1\n    for i in range(2, n + 1):\n        r *= i\n    return r'},
        {'question': 'Comment lire un fichier JSON en Python ?',
         'reponse': 'import json\nwith open("d.json", encoding="utf-8") as f:\n    data = json.load(f)'},
        {'question': "Écris une fonction qui retire les doublons d'une liste en gardant l'ordre.",
         'reponse': 'def uniques(xs):\n    vus = set()\n    return [x for x in xs if not (x in vus or vus.add(x))]'},
        {'question': 'Comment trier un dictionnaire par valeur ?',
         'reponse': 'sorted(d.items(), key=lambda kv: kv[1])'},
        {'question': 'Écris un compteur de mots en Python.',
         'reponse': 'from collections import Counter\nCounter(texte.split())'},
    ],
    'general': [
        {'question': "Explique en deux phrases ce qu'est un LLM.",
         'reponse': "Un LLM est un modèle statistique entraîné sur d'énormes quantités de texte pour prédire le mot suivant. Il ne « comprend » pas au sens humain : il calcule la suite la plus probable."},
        {'question': 'Donne deux avantages du calcul en local.',
         'reponse': "Vie privée (rien ne sort de la machine) et disponibilité (aucune connexion requise)."},
        {'question': "Qu'est-ce que le surapprentissage ?",
         'reponse': "Le surapprentissage, c'est quand un modèle apprend le bruit des données d'entraînement et généralise mal sur des données nouvelles."},
        {'question': 'Explique la différence entre RAM et disque.',
         'reponse': "La RAM est rapide et volatile (perdue à l'extinction), le disque est lent et persistant."},
        {'question': 'Pourquoi quantifier un modèle ?',
         'reponse': "Pour réduire sa taille en mémoire et accélérer l'inférence, au prix d'une légère perte de précision."},
        {'question': 'Cite deux règles pour écrire du code lisible.',
         'reponse': 'Des noms explicites, et des fonctions courtes qui font une seule chose.'},
    ],
}

def _ecrire(cat, exemples):
    d = Path('datasets'); d.mkdir(exist_ok=True)
    with open(d / f'{cat}.jsonl', 'w', encoding='utf-8') as f:
        for e in exemples:
            f.write(json.dumps(e, ensure_ascii=False) + '\n')

def _lignes(p):
    return [json.loads(l) for l in Path(p).read_text(encoding='utf-8').splitlines() if l.strip()]

source = {}
for cat in CATEGORIES:
    local = Path('datasets') / f'{cat}.jsonl'
    if local.is_file() and len(_lignes(local)) >= 5:
        source[cat] = f'local ({len(_lignes(local))} ex.)'
        continue
    source[cat] = ''

# pas de fichiers locaux complets -> on tente le depot, sinon l'amorce
if not all(source.values()):
    if not Path('aura-1b').exists():
        print('Clone du depot (pour les datasets)...')
        subprocess.run(['git', 'clone', '--depth', '1', DEPOT], check=False)
    for cat in CATEGORIES:
        if source[cat]:
            continue
        src = Path('aura-1b') / 'datasets' / f'{cat}.jsonl'
        if src.is_file():
            _ecrire(cat, _lignes(src))
            source[cat] = f'depot ({len(_lignes(src))} ex.)'
        else:
            _ecrire(cat, AMORCE[cat])
            source[cat] = f'amorce ({len(AMORCE[cat])} ex.)'

for cat in CATEGORIES:
    n = len(_lignes(Path('datasets') / f'{cat}.jsonl'))
    note = '' if n >= 50 else '  <-- petit : ajoute de vraies donnees pour un meilleur resultat'
    print(f'{cat:8s} {n:4d} exemples   source={source[cat]}{note}')


## 🏋️ Entraînement des 4 adaptateurs

La cellule est **idempotente** : relancer l'entraînement réutilise un adaptateur
déjà présent dans `tmp-lora-<cat>/final` (pas de retravail de `web`/`math`/`code`).
La mémoire GPU est libérée entre chaque catégorie (`del` + `gc.collect()` +
`torch.cuda.empty_cache()`) et l'empreinte par catégorie est minimale
(microbatch 1, `max_length` 512) : le pic tient dans la T4 de 14,5 Gio.

Le template appliqué est **exactement** celui du cerveau :
`<bos><start_of_turn>user\n{systeme}\n\n{contenu}<end_of_turn>\n<start_of_turn>model\n…<end_of_turn>`
Le système est préfixé au **premier tour utilisateur** (convention Gemma-3), donc
l'adaptateur apprend dans les mêmes conditions que l'inférence locale.


In [ ]:
# 3. TOUT le code d'entraînement (plus aucun .py a recuperer)
import json, subprocess, sys
from pathlib import Path

#: base d'entrainement = base du GGUF Heretic, en miroir NON gate
#: (google/gemma-3-1b-it exige un token HF + acceptation de licence).
BASE = 'unsloth/gemma-3-1b-it'
NOMS = {'web': 'aura-web', 'math': 'aura-math',
        'code': 'aura-code', 'general': 'aura-general'}

#: Personnalites IDENTIQUES a aura/agents.py (_PERSONNALITES).
PROMPT_PAR_CATEGORIE = {
    'web': ("Tu es un journaliste factuel. Reponds UNIQUEMENT a partir des "
            "faits fournis (WEB / BASE DE FAITS) ; si l'information manque, "
            "dis-le franchement. Cite chiffres et dates."),
    'math': ("Tu es un expert en calcul et raisonnement numerique. "
             "Reponds droit au but, montre le calcul en une ligne."),
    'code': ("Tu es un expert Python. Donne du code propre et commente, "
             "avec un exemple d'utilisation en commentaire."),
    'general': ("Tu es un assistant clair et precis. Structure ta reponse, "
                "va a l'essentiel, n'invente aucun fait precis."),
}

# Balises Gemma-3, assemblees a partir de morceaux (jamais confondues avec du HTML).
_B = "<"
EOT = _B + "end_of_turn>"
SOT = _B + "start_of_turn>"
BOS = _B + "bos>"
OUV_PENSEE = _B + "think>"
FER_PENSEE = _B + "/think>"

TEMPLATE = (
    BOS
    + SOT + "user\n{systeme}\n\n{contenu}" + EOT + "\n"
    + SOT + "model\n{corps}" + EOT + "\n"
)

def corps_reponse(reponse, pensee=None):
    """Tour `model` : bloc de pensee optionnel + reponse finale."""
    if pensee:
        return (OUV_PENSEE + pensee.strip() + FER_PENSEE + "\n"
                + reponse.strip())
    return reponse.strip()

def charger_dataset(chemin, categorie):
    """Lit le JSONL et applique la personnalite de la categorie."""
    systeme = PROMPT_PAR_CATEGORIE[categorie]
    exemples = []
    with open(chemin, encoding='utf-8') as f:
        for ligne in f:
            ligne = ligne.strip()
            if not ligne:
                continue
            e = json.loads(ligne)
            contenu = e['question']
            if e.get('contexte'):
                contenu = f"FAITS WEB :\n{e['contexte']}\n\n{contenu}"
            exemples.append({'texte': TEMPLATE.format(
                systeme=systeme, contenu=contenu,
                corps=corps_reponse(e['reponse'], e.get('pensee')))})
    return exemples

def _importer_peft():
    """Importe peft en neutralisant un torchao incompatible (cf. cellule 1)."""
    try:
        from peft import LoraConfig, get_peft_model
        return LoraConfig, get_peft_model
    except ImportError as err:
        if 'torchao' not in str(err):
            raise
        print('[fix] torchao incompatible avec peft -> desinstallation')
        subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y',
                        'torchao'], check=True, capture_output=True)
        from peft import LoraConfig, get_peft_model
        return LoraConfig, get_peft_model

def _dtype(torch):
    """bfloat16 si le GPU le gere (Ampere+), sinon float16 (T4 = Turing)."""
    try:
        if torch.cuda.is_available() and torch.cuda.is_bf16_supported():
            return torch.bfloat16
    except Exception:
        pass
    return torch.float16

class _DatasetTensors:
    """Adapte un BatchEncoding a l'API Dataset du Trainer.

    Les transformers recents exigent une vraie classe Dataset (indexation
    entiere) ; le dict brut de tenseurs leve KeyError: 0 au batching.
    """

    def __init__(self, enc):
        self._enc = enc

    def __len__(self):
        return len(self._enc['input_ids'])

    def __getitem__(self, i):
        return {k: v[i] for k, v in self._enc.items()}

def entrainer(categorie, dataset, epochs=3, sortie='adaptateurs', r=16,
              max_length=512):
    import gc, torch
    from transformers import (AutoModelForCausalLM, AutoTokenizer,
                              TrainingArguments, Trainer,
                              DataCollatorForLanguageModeling)
    LoraConfig, get_peft_model = _importer_peft()

    # Le convertisseur GGUF de llama.cpp ne cree PAS le dossier de sortie :
    # on le cree ici pour que la conversion trouve sa cible.
    Path(sortie).mkdir(exist_ok=True)

    # Idempotence : un adaptateur deja entraine (ex. web/math/code d'un run
    # precedent) est reutilise tel quel, sans recharger le modele.
    final = Path(f'./tmp-lora-{categorie}/final')
    if (final / 'adapter_config.json').is_file():
        print(f'[SKIP] {categorie} : adaptateur deja present ({final})')
        return Path(sortie) / f'{NOMS[categorie]}.gguf'

    # Libere la memoire GPU de la categorie precedente AVANT de recharger.
    # Sans ca, chaque modele reste reserve -> OOM a la 4e categorie sur T4.
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
        libre, total = torch.cuda.mem_get_info()
        print(f'[GPU] {libre / 2**30:.1f} Gio libres / {total / 2**30:.1f} Gio')

    tok = AutoTokenizer.from_pretrained(BASE)
    tok.pad_token = tok.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        BASE, torch_dtype=_dtype(torch), device_map='auto')

    # LoRA : ~10-40 Mo exporte ; cible les projections d'attention
    # (memes noms de modules chez Gemma-3 : q_proj/k_proj/v_proj/o_proj).
    config = LoraConfig(
        r=r, lora_alpha=2 * r, lora_dropout=0.05,
        target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
        task_type='CAUSAL_LM')
    model = get_peft_model(model, config)
    model.print_trainable_parameters()

    exemples = charger_dataset(dataset, categorie)
    assert len(exemples) >= 10, 'dataset trop petit (>= 10 exemples)'
    ds = _DatasetTensors(tok([e['texte'] for e in exemples], truncation=True,
                             max_length=max_length, padding=True,
                             return_tensors='pt'))

    # Empreinte minimale : microbatch 1 + accumulation (batch effectif 8),
    # max_length 512. Le pic memoire d'une categorie tient alors largement
    # dans les 14,5 Gio d'une T4.
    args = TrainingArguments(
        output_dir=f'./tmp-lora-{categorie}',
        num_train_epochs=epochs,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        learning_rate=2e-4,
        fp16=True, logging_steps=10,
        save_strategy='no', report_to=[])

    trainer = Trainer(model=model, args=args, train_dataset=ds,
                      data_collator=DataCollatorForLanguageModeling(tok, mlm=False))
    try:
        trainer.train()
        # Sauvegarde AVANT toute liberation (le LoRA seul fait ~12 Mo).
        model.save_pretrained(f'./tmp-lora-{categorie}/final')
        print(f'[OK] adapter PEFT : ./tmp-lora-{categorie}/final')
    finally:
        del trainer
        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    return Path(sortie) / f'{NOMS[categorie]}.gguf'


In [ ]:
# 4. Boucle d'entraînement : 4 catégories, ~5 min chacune sur T4
import gc, torch
# Nettoie tout residu GPU (ex. run precedent interrompu par l'OOM).
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    print(f'[GPU] {torch.cuda.memory_allocated() / 2**30:.1f} Gio deja alloues')
resultats = {}
for cat in ['web', 'math', 'code', 'general']:
    print('\n' + '=' * 60 + f'\n[CAT] {cat}\n' + '=' * 60)
    resultats[cat] = entrainer(cat, f'datasets/{cat}.jsonl', epochs=3)
print('\n[OK] Entraînements terminés :', resultats)


## 🔧 Conversion PEFT → GGUF

`llama.cpp` lit les adaptateurs au format GGUF (`--lora`). `--base-model-id`
donne l'architecture de référence ; `--outtype q8_0` garde un fichier petit.


In [ ]:
# 5. Conversion PEFT -> GGUF LoRA (format lu par llama.cpp)
!git clone --depth 1 --filter=blob:none --sparse https://github.com/ggml-org/llama.cpp.git tmp-llama-cpp
%cd tmp-llama-cpp
!git sparse-checkout set conversion gguf-py --skip-checks
%cd ..

import subprocess, os
from pathlib import Path
env = dict(os.environ, PYTHONPATH='tmp-llama-cpp/gguf-py')
Path('adaptateurs').mkdir(exist_ok=True)  # llama.cpp ne cree pas le dossier
for cat in ['web', 'math', 'code', 'general']:
    if not (Path(f'tmp-lora-{cat}/final') / 'adapter_config.json').is_file():
        print(f'[{cat}] ignore : adaptateur absent (tmp-lora-{cat}/final)')
        continue
    cmd = ['python', 'tmp-llama-cpp/convert_lora_to_gguf.py',
           '--outfile', f'adaptateurs/aura-{cat}.gguf',
           '--outtype', 'q8_0',
           '--base-model-id', BASE,
           f'tmp-lora-{cat}/final']
    r = subprocess.run(cmd, env=env, capture_output=True, text=True)
    print(f'[{cat}]', 'OK' if r.returncode == 0 else r.stderr[-800:])
!ls -lh adaptateurs/*.gguf


## 📥 Récupérer les adaptateurs

Le zip contient les 4 fichiers `.gguf`. Extrais-le et dépose les 4 fichiers dans
le dossier `adaptateurs/` du projet sur ton PC :

```
C:/Users/admin/Desktop/Dev/IA/aura-1b/adaptateurs/
├── aura-web.gguf
├── aura-math.gguf
├── aura-code.gguf
└── aura-general.gguf
```

Puis lance RontoGPT avec `AURA_ADAPTATEURS=1`. Le montage lit l'architecture **de
l'adaptateur lui-même** (son `general.architecture` GGUF) : les nouveaux `.gguf`
Gemma sont acceptés automatiquement, les anciens Llama restent refusés — aucune
ligne de code à changer.


In [ ]:
# 6. Zip final + téléchargement
!zip -j adaptateurs_aura_gemma.zip adaptateurs/*.gguf
from google.colab import files
files.download('adaptateurs_aura_gemma.zip')
